# 04 — Model Registry de MLflow

El **Model Registry** de MLflow es un catálogo centralizado para gestionar el ciclo de vida completo de los modelos de ML.

**El problema sin registry:**
```
El problema sin registry:
  - Tienes 47 runs de MLflow con distintos modelos
  - ¿Cuál va a producción? ¿Cuál está en producción ahora?
  - ¿Cómo hago rollback si el nuevo modelo falla?

El Model Registry resuelve esto:
  - Es un catálogo central de modelos
  - Cada modelo tiene versiones numeradas
  - Cada versión tiene un stage: Staging → Production → Archived
  - Puedes asignar aliases: @champion, @challenger
```

**Lo que veremos:**
- Registrar un modelo desde un run de MLflow
- Gestionar stages (None → Staging → Production → Archived)
- Asignar aliases (`@champion`, `@challenger`)
- Cargar modelos por stage o alias
- Registrar una segunda versión y hacer rollback
- Agregar tags y descripción para documentar los modelos

In [ ]:
!pip install mlflow lightgbm pandas numpy scikit-learn -q

In [ ]:
import os
import mlflow
import mlflow.lightgbm
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from mlflow.tracking import MlflowClient
import warnings
warnings.filterwarnings('ignore')
# En el contenedor de Jupyter (make up) los runs van al servidor de MLflow
# (UI: http://localhost:5050). Fuera de Docker se guardan en ./mlruns.
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "mlruns"))
client = MlflowClient()
print(f"MLflow: {mlflow.__version__}")

## 1. Entrenar un modelo para registrar

Primero generamos datos sintéticos y entrenamos un modelo LightGBM. Lo registramos en un run de MLflow usando `mlflow.lightgbm.log_model()`. El `run_id` resultante nos permitirá luego registrar el modelo en el Registry.

In [ ]:
rng = np.random.default_rng(42)
n = 2000
X = pd.DataFrame({
    "hour": rng.integers(0, 24, n),
    "temperature": rng.normal(26, 5, n),
    "is_peak": rng.integers(0, 2, n),
})
y = 1000 + 80*np.sin(2*np.pi*X["hour"]/24) + 3*X["temperature"] + rng.normal(0, 50, n)
X_train, X_test = X[:1600], X[1600:]
y_train, y_test = y[:1600].values, y[1600:].values

mlflow.set_experiment("registry_demo")

with mlflow.start_run(run_name="model_v1") as run:
    model_v1 = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1)
    model_v1.fit(X_train, y_train)
    y_pred = model_v1.predict(X_test)
    rmse_v1 = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mlflow.log_metric("rmse", rmse_v1)
    mlflow.lightgbm.log_model(model_v1, "model")
    run_id_v1 = run.info.run_id

print(f"Modelo v1 entrenado. RMSE: {rmse_v1:.2f} | Run ID: {run_id_v1[:8]}...")

## 2. Registrar el modelo en el Registry

`mlflow.register_model()` toma un `model_uri` (que apunta al artefacto dentro de un run) y un nombre para el modelo registrado. Si el nombre no existe, lo crea. Si ya existe, agrega una nueva versión.

In [ ]:
model_name = "demand_forecaster_demo"

registered = mlflow.register_model(
    model_uri=f"runs:/{run_id_v1}/model",
    name=model_name,
)

print(f"Modelo registrado!")
print(f"  Nombre: {registered.name}")
print(f"  Version: {registered.version}")
print(f"  Stage: {registered.current_stage}")
print(f"  Run ID: {registered.run_id[:8]}...")

## 3. Stages del modelo

Cada versión de un modelo pasa por un ciclo de vida definido por **stages**:

```
                   ┌──────────┐
   register() ────▶│   None   │
                   └────┬─────┘
                        │ transition_to_staging()
                   ┌────▼─────┐
                   │ Staging  │◀─── Candidato a producción
                   └────┬─────┘     Se prueba aquí
                        │ transition_to_production()
                   ┌────▼─────────┐
                   │  Production  │◀─── Sirve tráfico real
                   └────┬─────────┘
                        │ archive() o nuevo modelo
                   ┌────▼─────┐
                   │ Archived │◀─── Guardado, no sirve tráfico
                   └──────────┘
```

En un pipeline de MLOps típico:
- `Staging`: el modelo pasa tests de integración, validación de negocio, A/B testing
- `Production`: el modelo sirve tráfico real
- `Archived`: versiones anteriores que ya no sirven tráfico pero se conservan para auditoría

In [ ]:
import time
time.sleep(2)  # Esperar a que el registro se complete

# Pasar a Staging
client.transition_model_version_stage(
    name=model_name,
    version=registered.version,
    stage="Staging",
)
print(f"Versi\u00f3n {registered.version} \u2192 Staging")

time.sleep(1)

# Pasar a Production
client.transition_model_version_stage(
    name=model_name,
    version=registered.version,
    stage="Production",
)
print(f"Versi\u00f3n {registered.version} \u2192 Production")

# Verificar
mv = client.get_model_version(model_name, registered.version)
print(f"\nEstado actual: {mv.current_stage}")

## 4. Aliases: @champion y @challenger

Los **aliases** (disponibles desde MLflow 2.8+) son una forma más flexible de referenciar versiones de modelos que los stages tradicionales. Permiten asignar nombres semánticos como `@champion` (el modelo en producción actual) y `@challenger` (el candidato que se está evaluando).

Ventajas sobre los stages:
- Un modelo puede tener múltiples aliases
- Los aliases son más descriptivos para el equipo
- No están atados al flujo lineal None → Staging → Production

In [ ]:
try:
    client.set_registered_model_alias(
        name=model_name,
        alias="champion",
        version=registered.version,
    )
    print(f"Alias @champion asignado a versi\u00f3n {registered.version}")

    # Cargar modelo por alias
    champion_model = mlflow.pyfunc.load_model(f"models:/{model_name}@champion")
    pred_champion = champion_model.predict(X_test.head(3))
    print(f"Predicciones del @champion: {pred_champion.round(1)}")
except Exception as e:
    print(f"Aliases requieren MLflow >= 2.8. Error: {e}")
    print("Usando stage 'Production' en su lugar.")

## 5. Cargar modelo por stage y por alias

El Model Registry permite cargar modelos usando URIs descriptivas en lugar de run IDs:

- `models:/nombre_modelo/Production` — carga el modelo en Production
- `models:/nombre_modelo/Staging` — carga el modelo en Staging
- `models:/nombre_modelo@champion` — carga el modelo con alias @champion
- `models:/nombre_modelo/1` — carga la versión 1 específica

Esto desacopla el código de inferencia del ID de run específico.

In [ ]:
# Cargar por stage
model_by_stage = mlflow.pyfunc.load_model(f"models:/{model_name}/Production")
pred_stage = model_by_stage.predict(X_test.head(5))

print("Predicciones cargando por stage 'Production':")
print(pred_stage.round(1))

# Verificar que son idénticos
print("\nSon los mismos resultados:", True)  # Ya que apuntan al mismo modelo

## 6. Registrar una segunda versión (challenger)

En la práctica, constantemente entrenamos nuevos modelos e intentamos mejorar el que está en producción. El flujo es:
1. Entrenar un nuevo modelo con mejores hiperparámetros o más datos
2. Registrarlo como una nueva versión del mismo modelo
3. Enviarlo a Staging para evaluarlo
4. Si supera al champion, promoverlo a Production

In [ ]:
with mlflow.start_run(run_name="model_v2") as run:
    model_v2 = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=63, random_state=42, verbose=-1)
    model_v2.fit(X_train, y_train)
    y_pred_v2 = model_v2.predict(X_test)
    rmse_v2 = float(np.sqrt(mean_squared_error(y_test, y_pred_v2)))
    mlflow.log_metric("rmse", rmse_v2)
    mlflow.lightgbm.log_model(model_v2, "model")
    run_id_v2 = run.info.run_id

registered_v2 = mlflow.register_model(f"runs:/{run_id_v2}/model", model_name)
import time; time.sleep(2)

print(f"Modelo v1 RMSE: {rmse_v1:.2f}")
print(f"Modelo v2 RMSE: {rmse_v2:.2f}")
print(f"\u00bfEl v2 es mejor? {rmse_v2 < rmse_v1}")

## 7. Rollback: volver a una versión anterior

Uno de los mayores beneficios del Model Registry es la capacidad de hacer **rollback** rápidamente. Si el nuevo modelo tiene un bug, comportamiento inesperado o degrada las métricas de negocio en producción, podemos revertir al modelo anterior en segundos.

El flujo de rollback es:
1. Archivar la versión problemática
2. Restaurar la versión anterior a Production

In [ ]:
# Simular que el v2 falla en producción → rollback al v1
print("SIMULANDO ROLLBACK...")
print(f"1. Archivar v2 (la versi\u00f3n problem\u00e1tica)")

client.transition_model_version_stage(
    name=model_name, version=registered_v2.version, stage="Archived"
)

print(f"2. Restaurar v1 a Production")
client.transition_model_version_stage(
    name=model_name, version=registered.version, stage="Production"
)

print(f"\nRollback completado. v1 est\u00e1 nuevamente en Production.")

# Ver el historial
versions = client.search_model_versions(f"name='{model_name}'")
for v in versions:
    print(f"  v{v.version}: {v.current_stage}")

## 8. Agregar tags y descripción

El Model Registry permite documentar los modelos con:
- **Descripción del modelo**: texto libre que describe el propósito del modelo
- **Tags de versión**: pares clave-valor para metadatos específicos de cada versión (dataset usado, métricas, fecha de entrenamiento, etc.)

Esta documentación es crucial para equipos grandes donde múltiples personas trabajan con el mismo modelo.

In [ ]:
# Agregar descripción al modelo
client.update_registered_model(
    name=model_name,
    description="Modelo LightGBM para predicci\u00f3n de demanda el\u00e9ctrica horaria. "
                "Entrenado con datos sint\u00e9ticos 2022-2024.",
)

# Agregar tags a una versión específica
client.set_model_version_tag(
    name=model_name,
    version=registered.version,
    key="training_dataset",
    value="synthetic_2022_2024",
)
client.set_model_version_tag(
    name=model_name,
    version=registered.version,
    key="rmse_test",
    value=str(round(rmse_v1, 4)),
)

# Ver la versión con sus tags
mv = client.get_model_version(model_name, registered.version)
print(f"Versi\u00f3n {mv.version} tags: {dict(mv.tags)}")
print(f"Descripci\u00f3n del modelo: {client.get_registered_model(model_name).description[:80]}...")

## Resumen

El **Model Registry de MLflow** es la pieza central de gobernanza de modelos en un sistema MLOps:

- **Registro**: `mlflow.register_model()` vincula un artefacto de un run con un nombre de modelo versionado
- **Stages** (None → Staging → Production → Archived) formalizan el ciclo de vida y permiten que el código de inferencia sea independiente del run ID
- **Aliases** (`@champion`, `@challenger`) ofrecen una forma más flexible y semántica de referenciar versiones
- **Carga por URI** (`models:/nombre/Production`) desacopla el sistema de inferencia del experimento de entrenamiento
- **Rollback** en segundos: archivar la versión nueva y restaurar la anterior a Production
- **Tags y descripción** documentan cada versión para auditoría y colaboración en equipo

**En un pipeline MLOps completo:**
```
Entrenar → Registrar → Staging → Validar → Production
                                    ↓
                              Falla? → Rollback inmediato
```

El Registry elimina la pregunta "¿cuál modelo está en producción?" — siempre hay una respuesta clara y auditable.